In [1]:
import os
import pandas as pd

from langchain_ollama import ChatOllama
from src.graphs.ClassificationGraph import build_classification_graph
from datetime import datetime

In [2]:
MAX_TRIALS = 10
model = ChatOllama(
    model="nemotron-3-ultra:cloud",
    temperature=0,
    reasoning=False
)
#gemma4:31b-cloud

In [3]:
# 1. Define the Graph
model_name = "gemma4:31b-cloud"
phrases = {
    "lib_phrases" : "library, loan items, customers, member, membership card, member number, details, name, address, date of birth, subject sections, classification mark, bar code, language tapes, books, title language, level, title, author(s), current loan, bar code reader, membership, book bar code, records, number of subject sections, types of loan items, number of items, update of records, issues, shows, kept, denoted, borrow, reserved, renewed, extend, scanned, entered, read, stamped, searched, identified, French, beginner, daily, valid, two, maximum of 8, less than 8, a number of, has a title language, has a title and author(s), made up of, customer is known as a member, language tapes and books are two types of loan items",

    "ntss_phrases" : "NTSS, service provider, business customers, create, promote, organize, run, national, international, trade shows, contact, services, involves, design, including, theme, slogan, location, duration, promoting, advertisement, organizing, creation, promotion, inviting, speakers, registering, participants, exhibitors, running, registration, setting up, booths, conference rooms, seminars, reception, distributing, trade show materials, creates, account, each, customer, record, service charges, payments received, account balances, trade show, can be regarded as, event, has, organizer, person, organization, contact information, website, belong to, one or more, domains, added, removed, attended by, types of, organization staff, invited and/or selected, observers, register, fee, prepares, runs, invited, give, keynote address, selected, evaluation, proposals, reviewed by, committee, reviewers, status, proposal, pending, review, accepted, rejected, exhibit, products, pay, size of the booth, large, medium, small, requested, rented, duration of the event, visit",

    "rental_phrases" : "vehicle, manufacturer, price class, rental price, available, not available, rented out, purchase, repair, maintenance, disposal, car, passenger car, makes of car, model of car, transmission, automatic, manual, two, four, doors, sedan, hatchback, options, additional charge, depreciation of the rental cars, location, taken from, returned to, other forms of vehicle, customer, select, rental plan, daily unlimited miles plan, weekend savings plan, reserving, reservation, time of reservation, period of time, in person, by phone, voided, salesperson, process, archive, reservation form, file cabinet, sign, contract, block reservation, make, invoice, opened, cover, one or more, rentals, checked out, pay, sent to, company, rental charge, credit card, processed, credit card processing company, several"
}
truths = {
    "lib_truth" : pd.read_csv('../ground_truths/library.csv'),
    "rent_truth" : pd.read_csv('../ground_truths/car_rental.csv'),
    "ntss_truth" : pd.read_csv('../ground_truths/ntss.csv')
}

workflow = build_classification_graph(
    model=model,
    phrases=phrases,
    truths=truths,
    max_trials=10
)
agent = workflow.compile()

# 2. Run the Optimizer
# Pulling initial prompt from the specified file
prompt_path = "../prompts/auto_prompt_evo/classification/t0/prompt.md"
try:
    with open(prompt_path, 'r', encoding='utf-8') as f:
        initial_prompt = f.read()
    print("Successfully loaded prompt from:", prompt_path)
except FileNotFoundError:
    print(f"Error: Could not find file at {prompt_path}. Please check the path.")
    initial_prompt = "Fallback classification prompt... <DOMAIN PHRASES>"

inputs = {
    "current_prompt": initial_prompt,
    "current_trial": 1,
    "lib_eval": {}, "rental_eval": {}, "ntss_eval": {}
}

print(f"\n[{datetime.now():%H:%M:%S}] Starting optimizer: {MAX_TRIALS} trials planned")
result = agent.invoke(inputs)
print(f"\n[{datetime.now():%H:%M:%S}] Optimizer finished after {result['current_trial'] - 1} completed trials")


Successfully loaded prompt from: ../prompts/auto_prompt_evo/classification/t0/prompt.md

[23:23:26] Starting optimizer: 10 trials planned
[23:23:26] Trial 1 | lib: classifying domain phrases...
[23:23:26] Trial 1 | lib: done (51 chars)
[23:23:26] Trial 1 | rental: classifying domain phrases...
[23:23:54] Trial 1 | rental: done (51 chars)
[23:23:54] Trial 1 | ntss: classifying domain phrases...
[23:23:54] Trial 1 | ntss: done (51 chars)

--- Evaluation Results (trial 1) ---
Lib: p-0.00 r-0.00 f1-0.00 (TP=0 FP=0 FN=27)
Rent: p-0.00 r-0.00 f1-0.00 (TP=0 FP=0 FN=66)
Ntss: p-0.00 r-0.00 f1-0.00 (TP=0 FP=0 FN=53)
[23:23:54] Trial 1 | optimize: asking model to improve the prompt...
[23:24:03] Trial 1 | optimize: done
[23:24:03] Trial 1 | checkpoint: saving optimized prompt to ../prompts/auto_prompt_evo/classification/t1/prompt.md
[23:24:03] Trial 2 | lib: classifying domain phrases...


KeyboardInterrupt: 